In [ ]:
# Repository-relative paths for the anonymized reproduction package.
import os
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analysis_code').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from within the repository tree.')

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
MODULE_DIR = REPO_ROOT / 'analysis_code' / '03_dtw_phenotypes'
EXTERNAL_DATA_ROOT = Path(os.environ.get('HEATPA_DATA_ROOT', REPO_ROOT / 'external_data'))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.scale as mscale
import matplotlib.transforms as mtransforms
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from pathlib import Path

# ======================================================
# 0. SVG 与字体设置
# ======================================================
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.unicode_minus'] = False


# ======================================================
# 1. 全局参数设置
# ======================================================

# --------------------- 输入输出路径 ---------------------
csv_path = str(MODULE_DIR / "data" / "shape_features_standardized.csv")

output_dir = MODULE_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# --------------------- 坐标轴与视觉映射因子 ---------------------
x_col = 'late_auc'
y_col = 'decay_slope'
color_col = 'cluster'

# 该字段已经是总人口，只修改图例名称，不修改映射字段
size_col = 'total_all_mean_avg'

# --------------------- 坐标轴标题 ---------------------
X_AXIS_LABEL = 'Late-lag cumulative response'
Y_AXIS_LABEL = 'Temporal response slope'

# --------------------- 点大小范围 ---------------------
size_min = 100
size_max = 1300

# --------------------- 图片设置 ---------------------
axis_box_size = 6.0

legend_space = 3.1

left_margin = 0.9
right_margin = 0.3
bottom_margin = 0.8
top_margin = 0.3

fig_width = left_margin + axis_box_size + legend_space + right_margin
fig_height = bottom_margin + axis_box_size + top_margin

figsize = (fig_width, fig_height)

dpi = 300
scatter_alpha = 0.75

# --------------------- 文字大小参数 ---------------------
zone_label_fontsize = 16
axis_label_fontsize = 16
axis_tick_fontsize = 14
legend_fontsize = 12
legend_title_fontsize = 13

# --------------------- 文字层级参数 ---------------------
text_zorder = 100
legend_zorder = 110

show_title = False
plot_title = f'{x_col} vs {y_col}'
title_fontsize = 16

# ======================================================
# 参考第二个代码的聚类颜色
# ======================================================
COLOR_C1 = "#B23A3A"      # C1 red
COLOR_C2 = "#E69F00"      # C2 orange
COLOR_C3 = "#67A9CF"      # C3 light blue
COLOR_C4 = "#1F4E8C"      # C4 dark blue

# --------------------- 图例控制参数 ---------------------
cluster_legend_marker_size = 8
size_legend_levels = 5

# 圆圈大小图例颜色
size_legend_color = COLOR_C4

# 只修改 legend 名称
size_legend_title = 'Total population'

# --------------------- 圆圈大小图例间距控制 ---------------------
# 适度拉开点大小图例，但不显得过于松散
SIZE_LEGEND_LABEL_SPACING = 1.45     # 圆圈之间的垂直间距，适中
SIZE_LEGEND_HANDLE_TEXT_PAD = 1.0    # 圆圈与数字文字之间的距离
SIZE_LEGEND_HANDLE_HEIGHT = 1.35     # 每个图例项占据的高度
SIZE_LEGEND_MARKER_SCALE = 0.80      # 图例圆圈缩放比例
# --------------------- 散点大小自然断裂法参数 ---------------------
size_class_method = 'jenks'
size_class_levels = size_legend_levels
size_class_labels_precision = 2

# 两个图例的位置
cluster_legend_anchor = (1.02, 1.00)
size_legend_anchor = (1.02, 0.62)

# --------------------- 分区与坐标范围控制 ---------------------
render_zone_bg = False

# ======================================================
# 坐标轴四边视觉留白控制
# 数值越大，散点越向中间收缩；数值越小，散点越铺满图框
# 推荐范围：0.04–0.12
# ======================================================
visual_padding_ratio = 0.10

# --------------------- cluster 固定颜色 ---------------------
cluster_color_map = {
    1: COLOR_C1,
    2: COLOR_C2,
    3: COLOR_C3,
    4: COLOR_C4
}

# --------------------- 五分区背景颜色 ---------------------
zone_color_map = {
    1: '#313695',
    2: '#74ADD1',
    3: '#E53935',
    4: '#FEE090',
    5: '#F46D43'
}

zone_bg_alpha = 0.18

# --------------------- 输出文件名 ---------------------
save_svg_path = output_dir / f'{x_col}_{y_col}_{color_col}_{size_col}_five_zone_scatter_color_matched_jenks_6x6.svg'
csv_save_path = output_dir / f'{x_col}_{y_col}_{color_col}_{size_col}_five_zone_result_color_matched_jenks_6x6.csv'


# ======================================================
# 2. 自定义 PiecewiseLinearScale
# ======================================================
class PiecewiseLinearScale(mscale.ScaleBase):
    name = 'piecewise'

    def __init__(self, axis, **kwargs):
        super().__init__(axis)
        self.points = kwargs.get('points', [0, 1])
        self.scaled_points = kwargs.get('scaled_points', [0, 1])

    def get_transform(self):
        return self.PiecewiseLinearTransform(self.points, self.scaled_points)

    def set_default_locators_and_formatters(self, axis):
        axis.set_major_locator(mticker.MaxNLocator(nbins=5, steps=[1, 2, 5, 10]))
        axis.set_major_formatter(mticker.ScalarFormatter())

    class PiecewiseLinearTransform(mtransforms.Transform):
        input_dims, output_dims, is_separable = 1, 1, True

        def __init__(self, points, scaled_points):
            super().__init__()
            self.points = points
            self.scaled_points = scaled_points

        def transform_non_affine(self, a):
            return np.interp(a, self.points, self.scaled_points)

        def inverted(self):
            return PiecewiseLinearScale.PiecewiseLinearTransform(
                self.scaled_points,
                self.points
            )


mscale.register_scale(PiecewiseLinearScale)


# ======================================================
# 2.1 Jenks 自然断裂法函数
# ======================================================
def jenks_breaks(values, n_classes):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    values = np.sort(values)

    if len(values) == 0:
        raise ValueError("用于自然断裂法分级的数据为空。")

    unique_values = np.unique(values)
    n_classes = min(n_classes, len(unique_values))

    if n_classes <= 1:
        return [float(values.min()), float(values.max())]

    n_data = len(values)

    mat1 = np.zeros((n_data + 1, n_classes + 1))
    mat2 = np.zeros((n_data + 1, n_classes + 1))

    for i in range(1, n_classes + 1):
        mat1[0, i] = 1
        mat2[0, i] = 0

        for j in range(1, n_data + 1):
            mat2[j, i] = np.inf

    for l in range(2, n_data + 1):
        s1 = 0.0
        s2 = 0.0
        w = 0.0

        for m in range(1, l + 1):
            i3 = l - m + 1
            val = values[i3 - 1]

            s2 += val * val
            s1 += val
            w += 1

            v = s2 - (s1 * s1) / w
            i4 = i3 - 1

            if i4 != 0:
                for j in range(2, n_classes + 1):
                    if mat2[l, j] >= (v + mat2[i4, j - 1]):
                        mat1[l, j] = i3
                        mat2[l, j] = v + mat2[i4, j - 1]

        mat1[l, 1] = 1
        mat2[l, 1] = v

    breaks = [0.0] * (n_classes + 1)
    breaks[n_classes] = float(values[-1])
    breaks[0] = float(values[0])

    k = n_data

    for j in range(n_classes, 1, -1):
        idx = int(mat1[k, j]) - 2
        breaks[j - 1] = float(values[idx])
        k = int(mat1[k, j] - 1)

    breaks = sorted(list(dict.fromkeys(breaks)))

    return breaks


def classify_by_breaks(values, breaks):
    values = np.asarray(values, dtype=float)
    breaks = np.asarray(breaks, dtype=float)

    inner_breaks = breaks[1:-1]
    class_id = np.digitize(values, inner_breaks, right=True)

    return class_id


def make_class_labels(breaks, precision=2):
    labels = []

    for i in range(len(breaks) - 1):
        left = breaks[i]
        right = breaks[i + 1]

        if i == 0:
            label = f"{left:.{precision}f}–{right:.{precision}f}"
        else:
            label = f">{left:.{precision}f}–{right:.{precision}f}"

        labels.append(label)

    return labels


def calc_piecewise_balanced_limits(
    data_min,
    data_max,
    data_center,
    center_ratio,
    visual_padding
):
    """
    在保持 piecewise center_ratio 不变的前提下，
    计算使左右/上下视觉留白近似相等的数据坐标范围。
    """

    max_padding = min(center_ratio, 1 - center_ratio) * 0.85
    p = min(visual_padding, max_padding)

    left_span = data_center - data_min
    right_span = data_max - data_center

    if np.isclose(left_span, 0):
        left_span = abs(data_center) if data_center != 0 else 1

    if np.isclose(right_span, 0):
        right_span = abs(data_center) if data_center != 0 else 1

    left_fraction = p / center_ratio
    left_pad = left_fraction * left_span / (1 - left_fraction)

    right_fraction = p / (1 - center_ratio)
    right_pad = right_fraction * right_span / (1 - right_fraction)

    axis_min = data_min - left_pad
    axis_max = data_max + right_pad

    return axis_min, axis_max, left_pad, right_pad


# ======================================================
# 3. 读取数据
# ======================================================
Data = pd.read_csv(csv_path, header=0)

print(list(Data))


# ======================================================
# 4. 检查列名是否存在
# ======================================================
required_cols = [x_col, y_col, size_col, color_col]
missing_cols = [col for col in required_cols if col not in Data.columns]

if missing_cols:
    raise ValueError(f'以下列名不存在：{missing_cols}')


# ======================================================
# 5. 数据清洗
# ======================================================
unique_required_cols = list(dict.fromkeys(required_cols))
plot_data = Data[unique_required_cols].copy()

for col in [x_col, y_col, size_col]:
    plot_data[col] = pd.to_numeric(plot_data[col], errors='coerce')

plot_data[color_col] = pd.to_numeric(plot_data[color_col], errors='coerce')

plot_data = plot_data.dropna(subset=unique_required_cols).copy()

plot_data[color_col] = plot_data[color_col].astype(int)


# ======================================================
# 6. 五分区计算
# ======================================================
x_mean = plot_data[x_col].mean()
y_mean = plot_data[y_col].mean()

conditions = [
    (plot_data[x_col] >= x_mean) & (plot_data[y_col] >= y_mean),
    (plot_data[x_col] >= x_mean) & (plot_data[y_col] < y_mean),
    (plot_data[x_col] < x_mean) & (plot_data[y_col] < y_mean),
    (plot_data[x_col] < x_mean) & (plot_data[y_col] >= y_mean)
]

plot_data['zone'] = np.select(
    conditions,
    [1, 2, 3, 4],
    default=0
)

densest_quadrant = plot_data['zone'].value_counts().idxmax()
original_zone_number = densest_quadrant

slope = -1 if densest_quadrant in [1, 3] else 1

split_condition = (
    plot_data[y_col] - y_mean
) > slope * (
    plot_data[x_col] - x_mean
)

if densest_quadrant in [3, 4]:
    plot_data.loc[
        (plot_data['zone'] == densest_quadrant) & (~split_condition),
        'zone'
    ] = 5
else:
    plot_data.loc[
        (plot_data['zone'] == densest_quadrant) & (split_condition),
        'zone'
    ] = 5


# ======================================================
# 7. 坐标轴比例控制与范围外扩
# ======================================================
x_data_min = plot_data[x_col].min()
x_data_max = plot_data[x_col].max()
y_data_min = plot_data[y_col].min()
y_data_max = plot_data[y_col].max()

x_range = x_data_max - x_data_min
y_range = y_data_max - y_data_min

if np.isclose(x_range, 0):
    x_range = abs(x_data_max) if x_data_max != 0 else 1

if np.isclose(y_range, 0):
    y_range = abs(y_data_max) if y_data_max != 0 else 1

# 保留原始 piecewise 分布逻辑
x_ratio = 0.70 if densest_quadrant in [3, 4] else 0.30
y_ratio = 0.70 if densest_quadrant in [2, 3] else 0.30

x_min, x_max, x_left_pad, x_right_pad = calc_piecewise_balanced_limits(
    data_min=x_data_min,
    data_max=x_data_max,
    data_center=x_mean,
    center_ratio=x_ratio,
    visual_padding=visual_padding_ratio
)

y_min, y_max, y_lower_pad, y_upper_pad = calc_piecewise_balanced_limits(
    data_min=y_data_min,
    data_max=y_data_max,
    data_center=y_mean,
    center_ratio=y_ratio,
    visual_padding=visual_padding_ratio
)


# ======================================================
# 8. 散点大小映射：Jenks 自然断裂法
# ======================================================
s_min = float(plot_data[size_col].min())
s_max = float(plot_data[size_col].max())

if np.isclose(s_max, s_min):
    size_breaks = [s_min, s_max]
    size_class_count = 1
    plot_data["size_class"] = 0
    plot_data["point_size"] = (size_min + size_max) / 2
    size_class_sizes = np.array([(size_min + size_max) / 2])
    size_class_labels = [f"{s_min:.{size_class_labels_precision}f}"]
else:
    size_breaks = jenks_breaks(
        plot_data[size_col].values,
        size_class_levels
    )

    size_class_count = len(size_breaks) - 1

    plot_data["size_class"] = classify_by_breaks(
        plot_data[size_col].values,
        size_breaks
    )

    plot_data["size_class"] = plot_data["size_class"].clip(
        lower=0,
        upper=size_class_count - 1
    )

    size_class_sizes = np.linspace(
        size_min,
        size_max,
        size_class_count
    )

    size_map = {
        i: size_class_sizes[i]
        for i in range(size_class_count)
    }

    plot_data["point_size"] = plot_data["size_class"].map(size_map)

    size_class_labels = make_class_labels(
        size_breaks,
        precision=size_class_labels_precision
    )


# ======================================================
# 9. 绘图
# ======================================================
fig = plt.figure(figsize=figsize, dpi=dpi)

ax = fig.add_axes([
    left_margin / fig_width,
    bottom_margin / fig_height,
    axis_box_size / fig_width,
    axis_box_size / fig_height
])

ax.set_box_aspect(1)

ax.set_xscale(
    'piecewise',
    points=[x_min, x_mean, x_max],
    scaled_points=[0, x_ratio, 1]
)

ax.set_yscale(
    'piecewise',
    points=[y_min, y_mean, y_max],
    scaled_points=[0, y_ratio, 1]
)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)


# ======================================================
# 10. 绘制五分区背景
# ======================================================
if render_zone_bg:
    ax.add_patch(
        patches.Rectangle(
            (x_ratio, y_ratio),
            1 - x_ratio,
            1 - y_ratio,
            transform=ax.transAxes,
            color=zone_color_map[1],
            alpha=zone_bg_alpha,
            zorder=-3
        )
    )

    ax.add_patch(
        patches.Rectangle(
            (x_ratio, 0),
            1 - x_ratio,
            y_ratio,
            transform=ax.transAxes,
            color=zone_color_map[2],
            alpha=zone_bg_alpha,
            zorder=-3
        )
    )

    ax.add_patch(
        patches.Rectangle(
            (0, 0),
            x_ratio,
            y_ratio,
            transform=ax.transAxes,
            color=zone_color_map[3],
            alpha=zone_bg_alpha,
            zorder=-3
        )
    )

    ax.add_patch(
        patches.Rectangle(
            (0, y_ratio),
            x_ratio,
            1 - y_ratio,
            transform=ax.transAxes,
            color=zone_color_map[4],
            alpha=zone_bg_alpha,
            zorder=-3
        )
    )

    cp = [x_ratio, y_ratio]

    if densest_quadrant == 1:
        poly = [cp, [x_ratio, 1], [1, 1]]
    elif densest_quadrant == 2:
        poly = [cp, [1, y_ratio], [1, 0]]
    elif densest_quadrant == 3:
        poly = [cp, [0, 0], [0, y_ratio]]
    elif densest_quadrant == 4:
        poly = [cp, [0, 1], [x_ratio, 1]]

    ax.add_patch(
        patches.Polygon(
            poly,
            transform=ax.transAxes,
            color=zone_color_map[5],
            alpha=zone_bg_alpha + 0.08,
            zorder=-2
        )
    )


# ======================================================
# 11. 绘制散点
# ======================================================
groups = sorted(plot_data[color_col].unique())

for g in groups:
    sub = plot_data[plot_data[color_col] == g]
    point_color = cluster_color_map.get(g, '#BDBDBD')

    ax.scatter(
        sub[x_col],
        sub[y_col],
        s=sub['point_size'],
        c=point_color,
        alpha=scatter_alpha,
        edgecolors='none',
        linewidths=0,
        zorder=10
    )


# ======================================================
# 12. 绘制均值分割线与最密集象限斜向分割线
# ======================================================
line_style = {
    'color': 'dimgray',
    'linestyle': '--',
    'linewidth': 1.2,
    'alpha': 0.85,
    'zorder': 20
}

ax.axvline(x=x_mean, **line_style)
ax.axhline(y=y_mean, **line_style)

if densest_quadrant == 1:
    ax.plot([x_ratio, 1], [y_ratio, 1], transform=ax.transAxes, **line_style)
elif densest_quadrant == 2:
    ax.plot([x_ratio, 1], [y_ratio, 0], transform=ax.transAxes, **line_style)
elif densest_quadrant == 3:
    ax.plot([x_ratio, 0], [y_ratio, 0], transform=ax.transAxes, **line_style)
elif densest_quadrant == 4:
    ax.plot([x_ratio, 0], [y_ratio, 1], transform=ax.transAxes, **line_style)


# ======================================================
# 13. Zone 标签
# ======================================================
pad = 0.03

font_props = {
    'fontsize': zone_label_fontsize,
    'fontweight': 'normal',
    'color': 'black',
    'zorder': text_zorder
}

unsplit = {1, 2, 3, 4} - {original_zone_number}

for z in unsplit:
    if z == 1:
        ax.text(
            1 - pad, 1 - pad, 'Zone 1',
            transform=ax.transAxes,
            ha='right',
            va='top',
            **font_props
        )
    elif z == 2:
        ax.text(
            1 - pad, pad, 'Zone 2',
            transform=ax.transAxes,
            ha='right',
            va='bottom',
            **font_props
        )
    elif z == 3:
        ax.text(
            pad, pad, 'Zone 3',
            transform=ax.transAxes,
            ha='left',
            va='bottom',
            **font_props
        )
    elif z == 4:
        ax.text(
            pad, 1 - pad, 'Zone 4',
            transform=ax.transAxes,
            ha='left',
            va='top',
            **font_props
        )

if original_zone_number == 1:
    ax.text(
        x_ratio + pad, 1 - pad, 'Zone 1',
        transform=ax.transAxes,
        ha='left',
        va='top',
        **font_props
    )
    ax.text(
        1 - pad, y_ratio + pad, 'Zone 5',
        transform=ax.transAxes,
        ha='right',
        va='bottom',
        **font_props
    )

elif original_zone_number == 2:
    ax.text(
        1 - pad, y_ratio - pad, 'Zone 2',
        transform=ax.transAxes,
        ha='right',
        va='top',
        **font_props
    )
    ax.text(
        x_ratio + pad, pad, 'Zone 5',
        transform=ax.transAxes,
        ha='left',
        va='bottom',
        **font_props
    )

elif original_zone_number == 3:
    ax.text(
        x_ratio - pad, pad, 'Zone 3',
        transform=ax.transAxes,
        ha='right',
        va='bottom',
        **font_props
    )
    ax.text(
        pad, y_ratio - pad, 'Zone 5',
        transform=ax.transAxes,
        ha='left',
        va='top',
        **font_props
    )

elif original_zone_number == 4:
    ax.text(
        pad, y_ratio + pad, 'Zone 4',
        transform=ax.transAxes,
        ha='left',
        va='bottom',
        **font_props
    )
    ax.text(
        x_ratio - pad, 1 - pad, 'Zone 5',
        transform=ax.transAxes,
        ha='right',
        va='top',
        **font_props
    )


# ======================================================
# 14. 坐标轴、图名与图例
# ======================================================
ax.set_xlabel(
    X_AXIS_LABEL,
    fontsize=axis_label_fontsize
)

ax.set_ylabel(
    Y_AXIS_LABEL,
    fontsize=axis_label_fontsize
)

ax.xaxis.label.set_zorder(text_zorder)
ax.yaxis.label.set_zorder(text_zorder)

ax.tick_params(axis='both', labelsize=axis_tick_fontsize)

for tick_label in ax.get_xticklabels():
    tick_label.set_zorder(text_zorder)

for tick_label in ax.get_yticklabels():
    tick_label.set_zorder(text_zorder)

if show_title:
    title_obj = ax.set_title(plot_title, fontsize=title_fontsize)
    title_obj.set_zorder(text_zorder)


# ======================================================
# 14.1 Cluster 图例
# ======================================================
cluster_legend_handles = []

for g in groups:
    point_color = cluster_color_map.get(g, '#BDBDBD')

    cluster_legend_handles.append(
        Line2D(
            [0],
            [0],
            marker='o',
            linestyle='none',
            label=f'Cluster {g}',
            markerfacecolor=point_color,
            markeredgecolor='none',
            markersize=cluster_legend_marker_size,
            alpha=scatter_alpha
        )
    )

cluster_legend = ax.legend(
    handles=cluster_legend_handles,
    title='Cluster',
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
    frameon=False,
    loc='upper left',
    bbox_to_anchor=cluster_legend_anchor,
    borderaxespad=0
)

cluster_legend.set_zorder(legend_zorder)
ax.add_artist(cluster_legend)


# ======================================================
# 14.2 圆圈大小图例
# ======================================================
size_legend_handles = []

for label, size in zip(size_class_labels, size_class_sizes):
    size_legend_handles.append(
        Line2D(
            [0],
            [0],
            marker='o',
            linestyle='none',
            label=label,
            markerfacecolor=size_legend_color,
            markeredgecolor='none',
            markersize=np.sqrt(size) * SIZE_LEGEND_MARKER_SCALE,
            alpha=scatter_alpha
        )
    )

size_legend = ax.legend(
    handles=size_legend_handles,
    title=size_legend_title,
    fontsize=legend_fontsize,
    title_fontsize=legend_title_fontsize,
    frameon=False,
    loc='upper left',
    bbox_to_anchor=size_legend_anchor,
    borderaxespad=0,
    labelspacing=SIZE_LEGEND_LABEL_SPACING,
    handletextpad=SIZE_LEGEND_HANDLE_TEXT_PAD,
    handleheight=SIZE_LEGEND_HANDLE_HEIGHT
)

size_legend.set_zorder(legend_zorder)

ax.grid(False)


# ======================================================
# 15. 保存 SVG 图片与五分区结果
# ======================================================
plt.savefig(
    save_svg_path,
    format='svg',
    transparent=True
)

plt.show()

plot_data.to_csv(csv_save_path, index=False, encoding='utf-8-sig')

print(f'SVG 图片已保存至：{save_svg_path}')
print(f'五分区结果已保存至：{csv_save_path}')
print(f'整张图尺寸：{fig_width:.2f} × {fig_height:.2f} 英寸')
print(f'数据框尺寸：{axis_box_size:.2f} × {axis_box_size:.2f} 英寸')
print(f'横轴均值 {x_col} = {x_mean:.4f}')
print(f'纵轴均值 {y_col} = {y_mean:.4f}')
print(f'最密集象限为 Zone {original_zone_number}，其中一部分被进一步划分为 Zone 5')
print(f'x轴范围：{x_min:.4f} ~ {x_max:.4f}')
print(f'y轴范围：{y_min:.4f} ~ {y_max:.4f}')
print(f'x轴原始数据范围：{x_data_min:.4f} ~ {x_data_max:.4f}')
print(f'y轴原始数据范围：{y_data_min:.4f} ~ {y_data_max:.4f}')
print(f'视觉留白比例：{visual_padding_ratio}')
print(f'x轴数据留白：left={x_left_pad:.4f}, right={x_right_pad:.4f}')
print(f'y轴数据留白：lower={y_lower_pad:.4f}, upper={y_upper_pad:.4f}')
print(f'圆圈大小映射字段：{size_col}')
print(f'圆圈大小图例名称：{size_legend_title}')
print(f'圆圈大小分级方法：Jenks natural breaks')
print(f'圆圈大小自然断裂点：{size_breaks}')
print(f'实际圆圈大小等级数：{size_class_count}')